Qween

In [16]:
# =============================================================================
# Qwen2.5-VL medical expert over Group_1_Med_Data.xlsx  —  Google Colab script
# -----------------------------------------------------------------------------
# What it does:
#   1. Converts Group_1_Med_Data.xlsx  ->  med_data.json  (easy to handle)
#   2. For every row, builds a context from the patient fields + the user's
#      "Question" (the Facebook-group query) + the image(s) if present.
#   3. Asks Qwen2.5-VL-3B-Instruct (acting as a medical expert) a FIXED set of
#      5 medical questions for each row, and saves the answers to qwen_answers.json
#
# Image rules:
#   - A row may list 0, 1 or several image IDs in "Image_ID" (e.g. "img001, img002").
#   - Rows with NO image are processed as text-only.
#   - If a row DOES name an image but no matching file is found in images/,
#     the whole row is SKIPPED with a warning (per request).
#   - The images/ folder has a typo: img006 is saved as "img006.jgp". The
#     stem-based lookup below matches it regardless of extension.
#
# Run order in Colab:  upload the .xlsx and the images/ folder to /content,
# then run this whole file (e.g. `!python qwen_medical_colab.py` or paste cells).
# A GPU runtime is strongly recommended (Runtime > Change runtime type > T4 GPU).
# =============================================================================

# ----------------------------- 0. Install deps -------------------------------
# (In Colab, run these once. Lines are no-ops if already installed.)
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("openpyxl", "pandas", "pillow")
# Qwen3-VL needs a recent transformers (use the dev build to be safe).
_pip("git+https://github.com/huggingface/transformers", "accelerate")

# ------------------------------- 1. Config -----------------------------------
import os
# Reduce CUDA fragmentation (must be set BEFORE torch is imported anywhere).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json, glob, math, warnings
import pandas as pd

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# ================================================================
# QWEN MEDICAL DATA ANALYSIS — ONE CELL AFTER DRIVE MOUNT
# ================================================================

# ------------------------- INSTALL -------------------------------

!pip -q install -U transformers accelerate pandas openpyxl pillow


# ------------------------- IMPORTS --------------------------------

import os
import json
import math
import glob
import warnings
import gc

import pandas as pd
import torch
from PIL import Image


# ------------------------- CONFIG ---------------------------------

XLSX_PATH = "/content/drive/MyDrive/Bigg_Data/data/Group_7 Med Data 5.xlsx"

IMAGES_DIR = "/content/drive/MyDrive/Bigg_Data/data/images"

JSON_PATH = "/content/med_data.json"

ANSWERS_PATH = "/content/qwen_answers.json"

MODEL_ID = "Qwen/Qwen3.5-0.8B"

MAX_NEW_TOKENS = 384

# প্রথমে 1 দিয়ে test করো।
# কাজ ঠিক হলে 20 করো।
MAX_ROWS = 1

IMAGE_MAX_EDGE = 896

RESIZED_DIR = "/content/qwen_resized"


# ------------------------- DEVICE CHECK ---------------------------

print("=" * 70)
print("DEVICE CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    print("WARNING: GPU NOT AVAILABLE — running on CPU")
    DEVICE = "cpu"


# ------------------------- CHECK FILES ----------------------------

print("\n" + "=" * 70)
print("CHECKING FILES")
print("=" * 70)

if not os.path.exists(XLSX_PATH):
    raise FileNotFoundError(
        f"\nExcel file not found:\n{XLSX_PATH}"
    )

if not os.path.exists(IMAGES_DIR):
    raise FileNotFoundError(
        f"\nImages folder not found:\n{IMAGES_DIR}"
    )

print("[OK] Excel file found")
print("[OK] Images folder found")


# ------------------------- QUESTIONS ------------------------------

FIXED_QUESTIONS = [

    "1. Based on the information and image if provided, what is the most likely medical assessment of the described condition?",

    "2. What are the possible causes or contributing factors?",

    "3. What care steps, treatment, or management would you recommend?",

    "4. What warning signs mean the patient should see a doctor urgently?",

    "5. What preventive measures or general advice apply to this case?",

    "6. What is the expected recovery time or prognosis if the recommended treatment is followed?",

    "7. What treatment options are available, and how do they compare in terms of cost, duration, and effectiveness?",

    "8. What diagnostic tests or examinations would help confirm the assessment?",

    "9. What lifestyle, dietary, or hygiene changes should the patient make during and after treatment?",

    "10. What complications or risks could arise if the condition is left untreated or improperly managed?"

]

print(
    f"\n[OK] {len(FIXED_QUESTIONS)} questions configured"
)

print(
    f"Expected model calls for this run: "
    f"{MAX_ROWS * len(FIXED_QUESTIONS)}"
)


# ------------------------- CLEAN VALUE ----------------------------

def to_clean(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return None

        return value

    return value


# ------------------------- IMAGE IDS ------------------------------

def parse_image_ids(raw):

    if raw is None:
        return []

    raw = str(raw).strip()

    if not raw:
        return []

    parts = raw.replace(";", ",").split(",")

    return [
        p.strip()
        for p in parts
        if p.strip()
    ]


# ------------------------- EXCEL -> JSON --------------------------

print("\n" + "=" * 70)
print("READING EXCEL")
print("=" * 70)

df = pd.read_excel(XLSX_PATH)

print("Excel shape:", df.shape)

print("Columns:", list(df.columns))

records = []

for idx, row in df.iterrows():

    rec = {
        col: to_clean(row[col])
        for col in df.columns
    }

    rec["row_index"] = int(idx)

    rec["image_ids"] = parse_image_ids(
        rec.get("Image_ID")
    )

    records.append(rec)


with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"[OK] Wrote {len(records)} rows -> {JSON_PATH}"
)


# ------------------------- IMAGE INDEX ----------------------------

print("\n" + "=" * 70)
print("INDEXING IMAGES")
print("=" * 70)

image_index = {}

for path in glob.glob(
    os.path.join(IMAGES_DIR, "*")
):

    if not os.path.isfile(path):
        continue

    filename = os.path.basename(path)

    stem = os.path.splitext(
        filename
    )[0].lower().strip()

    image_index[stem] = path


print(
    f"[OK] Found {len(image_index)} image files"
)


# ------------------------- RESIZE IMAGES --------------------------

os.makedirs(
    RESIZED_DIR,
    exist_ok=True
)

resized_index = {}

print("\nPreparing images...")

for stem, path in image_index.items():

    try:

        img = Image.open(
            path
        ).convert("RGB")

        width, height = img.size

        longest = max(
            width,
            height
        )

        scale = min(
            1.0,
            IMAGE_MAX_EDGE / longest
        )

        if scale < 1.0:

            new_size = (
                max(1, int(width * scale)),
                max(1, int(height * scale))
            )

            img = img.resize(
                new_size,
                Image.Resampling.LANCZOS
            )

        output_path = os.path.join(
            RESIZED_DIR,
            stem + ".jpg"
        )

        img.save(
            output_path,
            "JPEG",
            quality=90
        )

        resized_index[stem] = output_path

    except Exception as e:

        warnings.warn(
            f"Could not process {path}: {e}"
        )


print(
    f"[OK] Prepared {len(resized_index)} images"
)


# ------------------------- PROMPT --------------------------------

SYSTEM_PROMPT = """
You are a careful medical information assistant.

You are given basic patient information, an optional medical image,
and a question originally posted by a user.

Provide a cautious, evidence-aware response.

Rules:

1. Do not claim certainty from an image alone.
2. If the image is unclear, explicitly say so.
3. Do not invent patient information.
4. Do not invent test results.
5. Distinguish possible conditions from confirmed diagnoses.
6. Recommend professional medical evaluation when appropriate.
7. Clearly mention urgent warning signs when relevant.
8. Answer in the SAME language as the user's question.
9. Keep the answer practical and understandable.
10. Do not provide false precision about cost or recovery time.
"""


# ------------------------- CONTEXT --------------------------------

def build_context_text(rec):

    fields = [
        ("Age", "Age"),
        ("Height", "Height"),
        ("Weight", "Weight"),
        ("Gender", "Gender"),
        ("Language", "Language")
    ]

    lines = []

    for col, label in fields:

        value = rec.get(col)

        if value is not None:

            lines.append(
                f"- {label}: {value}"
            )

    if lines:

        fields_block = "\n".join(lines)

    else:

        fields_block = (
            "- No structured patient details provided"
        )

    user_question = (
        rec.get("Question")
        or "(No question text provided)"
    )

    return fields_block, user_question


# ------------------------- USER MESSAGE ---------------------------

def build_user_message(
    rec,
    image_paths,
    fixed_question
):

    fields_block, user_question = (
        build_context_text(rec)
    )

    text = f"""
Patient details:

{fields_block}

Original user's question:

"{user_question}"

Specific question:

{fixed_question}

Answer this specific question using only the available
patient information and image.
"""

    content = []

    # Add images
    for image_path in image_paths:

        content.append({
            "type": "image",
            "image": image_path
        })

    # Add text
    content.append({
        "type": "text",
        "text": text.strip()
    })

    return content


# ------------------------- LOAD MODEL -----------------------------

print("\n" + "=" * 70)
print("LOADING QWEN MODEL")
print("=" * 70)

print(
    f"Model: {MODEL_ID}"
)

print(
    "First run may download the model..."
)

try:

    from transformers import (
        AutoProcessor,
        AutoModelForMultimodalLM
    )

except ImportError:

    raise ImportError(
        "Transformers installation failed. "
        "Restart Colab runtime and run this cell again."
    )


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print(
    "\n[OK] Qwen model loaded successfully."
)

print(
    "Model device:",
    model.device
)


# ------------------------- MEMORY --------------------------------

def free_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ------------------------- GENERATION -----------------------------

def answer_question(
    rec,
    image_paths,
    fixed_question
):

    messages = [

        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT
                }
            ]
        },

        {
            "role": "user",
            "content": build_user_message(
                rec,
                image_paths,
                fixed_question
            )
        }

    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = inputs.to(
        model.device
    )

    with torch.inference_mode():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    input_length = (
        inputs["input_ids"].shape[-1]
    )

    generated_answer = generated_ids[
        0
    ][input_length:]

    answer = processor.decode(
        generated_answer,
        skip_special_tokens=True
    )

    del inputs
    del generated_ids

    free_memory()

    return answer.strip()


# ------------------------- EXISTING RESULTS -----------------------

results = []

if os.path.exists(ANSWERS_PATH):

    try:

        with open(
            ANSWERS_PATH,
            "r",
            encoding="utf-8"
        ) as f:

            old_results = json.load(f)

        if isinstance(old_results, list):

            results = old_results

            print(
                f"\n[RESUME] Existing results: "
                f"{len(results)} rows"
            )

    except Exception as e:

        warnings.warn(
            f"Could not read existing output: {e}"
        )


completed_rows = {
    r.get("row_index")
    for r in results
    if isinstance(r, dict)
}


# ------------------------- SELECT ROWS ----------------------------

if MAX_ROWS is None:

    rows = records

else:

    rows = records[:MAX_ROWS]


# ------------------------- MAIN LOOP ------------------------------

print("\n" + "=" * 70)
print("STARTING QWEN ANALYSIS")
print("=" * 70)

processed = 0
skipped = 0

for rec in rows:

    ridx = rec["row_index"]

    # Already completed
    if ridx in completed_rows:

        print(
            f"\n[SKIP] Row {ridx} already completed."
        )

        continue


    # Resolve images
    image_paths = []
    missing = []

    for iid in rec["image_ids"]:

        stem = os.path.splitext(
            str(iid)
        )[0].lower().strip()

        if stem in resized_index:

            image_paths.append(
                resized_index[stem]
            )

        else:

            missing.append(iid)


    # Missing image
    if missing:

        warnings.warn(
            f"[SKIP] Row {ridx}: "
            f"missing image(s): {missing}"
        )

        skipped += 1

        continue


    kind = (
        "multimodal"
        if image_paths
        else
        "text-only"
    )

    print("\n" + "-" * 70)

    print(
        f"ROW {ridx} | {kind}"
    )

    print(
        "Images:",
        [
            os.path.basename(p)
            for p in image_paths
        ]
    )


    qa_pairs = []

    row_failed = False


    # 10 questions
    for qi, question in enumerate(
        FIXED_QUESTIONS,
        start=1
    ):

        print(
            f"  Q{qi}/{len(FIXED_QUESTIONS)} ..."
        )

        try:

            answer = answer_question(
                rec,
                image_paths,
                question
            )

            qa_pairs.append({

                "question": question,

                "answer": answer

            })

            print(
                "    [OK]"
            )

        except Exception as e:

            print(
                f"    [ERROR] "
                f"{type(e).__name__}: {e}"
            )

            row_failed = True

            break


    # Don't save incomplete row
    if row_failed:

        skipped += 1

        free_memory()

        continue


    # Build result
    result = {

        "row_index": ridx,

        "image_ids": rec["image_ids"],

        "images_used": [
            os.path.basename(p)
            for p in image_paths
        ],

        "patient": {

            "Age": rec.get("Age"),

            "Height": rec.get("Height"),

            "Weight": rec.get("Weight"),

            "Gender": rec.get("Gender"),

            "Language": rec.get("Language")

        },

        "user_question": rec.get(
            "Question"
        ),

        "qa_pairs": qa_pairs

    }


    results.append(result)

    completed_rows.add(ridx)

    processed += 1


    # Incremental save
    temp_path = ANSWERS_PATH + ".tmp"

    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            results,
            f,
            ensure_ascii=False,
            indent=2
        )

    os.replace(
        temp_path,
        ANSWERS_PATH
    )


    print(
        f"[SAVED] Row {ridx} -> "
        f"{ANSWERS_PATH}"
    )

    free_memory()


# ------------------------- FINAL CHECK ----------------------------

print("\n" + "=" * 70)
print("QWEN ANALYSIS FINISHED")
print("=" * 70)

print(
    f"Processed: {processed}"
)

print(
    f"Skipped: {skipped}"
)

print(
    f"Total saved rows: {len(results)}"
)

print(
    f"Output: {ANSWERS_PATH}"
)


# ------------------------- VERIFY JSON ----------------------------

if os.path.exists(ANSWERS_PATH):

    with open(
        ANSWERS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        final_data = json.load(f)

    print(
        "\n[OK] qwen_answers.json created successfully."
    )

    if final_data:

        print(
            "Rows in JSON:",
            len(final_data)
        )

        print(
            "Q&A pairs in first row:",
            len(
                final_data[0].get(
                    "qa_pairs",
                    []
                )
            )
        )

        print(
            "\nFirst answer:"
        )

        if final_data[0].get("qa_pairs"):

            print(
                final_data[0]
                ["qa_pairs"][0]
                ["answer"]
            )

else:

    print(
        "\n[ERROR] Output JSON was not created."
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 10.0 MB/s eta 0:00:00
DEVICE CHECK
PyTorch: 2.11.0+cpu
CUDA available: False

CHECKING FILES
[OK] Excel file found
[OK] Images folder found

[OK] 10 questions configured
Expected model calls for this run: 10

READING EXCEL
Excel shape: (20, 7)
Columns: ['Image_ID', 'Age', 'Height', 'Weight', 'Gender', 'Question', 'Language']
[OK] Wrote 20 rows -> /content/med_data.json

INDEXING IMAGES
[OK] Found 9 image files

Preparing images...
[OK] Prepared 9 images

LOADING QWEN MODEL
Model: Qwen/Qwen3.5-0.8B
First run may download the model...


ImportError: Transformers installation failed. Restart Colab runtime and run this cell again.

In [ ]:
!pip install -U transformers accelerate bitsandbytes openpyxl pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 70.8 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6a90436994308bdd0dd41d81', 'name': 'hasib20019', 'fullname': 'Hasibul Hasan Miazy', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/689fc3b4c521931d6a7b4659618c2558.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-04T06:51:35.000Z'}}


In [ ]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="google/gemma-3-4b-it",
    filename="config.json"
)

print("Gemma access SUCCESS!")
print(test_file)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

Gemma access SUCCESS!
/root/.cache/huggingface/hub/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767/config.json


In [ ]:
import os
import glob

XLSX_PATH = "/content/drive/MyDrive/Bigg_Data/data/Group_7 Med Data 5.xlsx"
IMAGES_DIR = "/content/drive/MyDrive/Bigg_Data/data/images"

print("Excel exists:", os.path.exists(XLSX_PATH))
print("Image folder exists:", os.path.exists(IMAGES_DIR))

images = glob.glob(os.path.join(IMAGES_DIR, "*"))

print("Number of image files:", len(images))

print("\nImages:")
for img in images:
    print(os.path.basename(img))

Excel exists: True
Image folder exists: True
Number of image files: 9

Images:
img_001.jpg
img_002.jpg
img_003.jpg
img_004.jpg
img_005.jpg
img_006.jpg
img_007.jpg
img_008.jpg
img_009.jpg


In [ ]:
import torch

from transformers import (
    AutoProcessor,
    Gemma3ForConditionalGeneration
)

MODEL_ID = "google/gemma-3-4b-it"

print("Loading Gemma 3 4B IT...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("================================")
print("Gemma loaded successfully!")
print("================================")

Loading Gemma 3 4B IT...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma loaded successfully!


In [ ]:
import pandas as pd

df = pd.read_excel(XLSX_PATH)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (20, 7)


,Image_ID,Age,Height,Weight,Gender,Question,Language
0,img_001.jpg,NaN,NaN,NaN,Male,"আমার দাঁতে এরকম গর্ত আছে, কিন্তু ব্যথা করে না।...",Bangla
1,NaN,NaN,NaN,NaN,Female,"ব্রেসেস এর ১১ মাস হয়ে গেছে,৪টা দাত তুলছি।দাত ত...",Bangla
2,img_002.jpg,NaN,NaN,NaN,Male,এইখানে নিচের পাটির ডান পাশের দাঁতটা রুট ক্যানে...,Bangla
3,NaN,NaN,NaN,NaN,NaN,দাঁতের ক্যাপ যদি খুলে যায় তাহলে ক্যাপটা পুনরা...,Bangla
4,NaN,NaN,NaN,NaN,Male,Pg hospital a ki faka daat er composite veneer...,transliterated


**Phi4 analysis**:


In [15]:
import torch
import os

print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("❌ GPU NOT AVAILABLE")

COLAB_GPU: 
PyTorch: 2.11.0+cpu
CUDA available: False
CUDA version: None
❌ GPU NOT AVAILABLE
COLAB_GPU: 
PyTorch: 2.11.0+cpu
CUDA available: False
CUDA version: None
❌ GPU NOT AVAILABLE
